# Recitation 0: Weights and Biases

In this recitation, you will learn about the importance of performance visualization and model tracking using [WandB](https://wandb.ai/), a tool for performance visualization, model and data version controlling and hyperparameter tuning.

## Installation and Libraries

In [1]:
## Installing WandB
!pip install wandb -qqq

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets
from torchvision.transforms import ToTensor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device: ", device)

from tqdm import tqdm

Device:  cuda


In [3]:
import wandb, os
# Authenticate with WANDB_API_KEY from the environment or the interactive login prompt.
os.environ['WANDB_API_KEY'] = "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"
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: 2792779840 (2792779840-zhejiang-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Helper functions and Model

In [4]:
DATA_ROOT = "/home/admin/CMU-11-785-s26/S26/documents/labs/lab_5/data"
data_train = datasets.CIFAR10(
    root = DATA_ROOT,
    train = True,
    transform = ToTensor(),
    download = True,
)
data_test = datasets.CIFAR10(
    root = DATA_ROOT,
    train = False,
    download = True,
    transform = ToTensor()
)

In [5]:
def build_data(batch_size, data_train, data_test):
    train_loader = torch.utils.data.DataLoader(data_train, batch_size=batch_size, shuffle=True)
    test_loader = torch.utils.data.DataLoader(data_test, batch_size=batch_size, shuffle=False)
    return train_loader, test_loader

In [6]:
class Network(nn.Module):

  def __init__(self):

    super(Network, self).__init__()

    self.CNN = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=64, kernel_size=3, stride=1, padding=2),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.AvgPool2d(kernel_size=9),
            nn.Flatten()
    )

    self.classification = nn.Linear(576, 10)
  def forward(self, x):

    x_cnn = self.CNN(x)
    res = self.classification(x_cnn)

    return res

model = Network().to(device)
print(model)

Network(
  (CNN): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(2, 2))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): AvgPool2d(kernel_size=9, stride=9, padding=0)
    (4): Flatten(start_dim=1, end_dim=-1)
  )
  (classification): Linear(in_features=576, out_features=10, bias=True)
)


In [7]:
train_loader, test_loader = build_data(64, data_train, data_test)

for x, y in train_loader:
  break
model(x.to(device)).shape

torch.Size([64, 10])

In [8]:
def get_optim(optimizer, learning_rate, model):
  if optimizer=='sgd':
    return optim.SGD(model.parameters(), lr=learning_rate)
  else:
    return optim.Adam(model.parameters(), lr=learning_rate)

In [9]:
def train_epoch(model, loader, optimizer, criterion, scaler):
    num_correct = 0
    total_loss = 0

    for i, (x, y) in enumerate(loader):
          optimizer.zero_grad()

          x = x.cuda()
          y = y.cuda()

          with torch.cuda.amp.autocast():
              outputs = model(x)
              loss = criterion(outputs, y)

          total_loss += float(loss)

          scaler.scale(loss).backward()
          scaler.step(optimizer)
          scaler.update()
    ep_loss = float(total_loss / len(loader))

    return model, ep_loss

In [10]:
def train(model, finish= True):

  # Dont worry about all this, you'll be very familiar with it after HW1

  best_acc = 0

  for epoch in range(run_config['epochs']):
      batch_bar = tqdm(total=len(train_loader), dynamic_ncols=True, leave=False, position=0, desc='Train')

      num_correct = 0
      total_loss = 0

      for i, (x, y) in enumerate(train_loader):
          optimizer.zero_grad()

          x = x.cuda()
          y = y.cuda()

          with torch.cuda.amp.autocast():
              outputs = model(x)
              loss = criterion(outputs, y)

          num_correct += int((torch.argmax(outputs, axis=1) == y).sum())
          total_loss += float(loss)

          batch_bar.set_postfix(
              acc="{:.04f}%".format(100 * num_correct / ((i + 1) * run_config['batch_size'])),
              loss="{:.04f}".format(float(total_loss / (i + 1))),
              num_correct=num_correct,
              lr="{:.04f}".format(float(optimizer.param_groups[0]['lr'])))

          scaler.scale(loss).backward()
          scaler.step(optimizer)
          scaler.update()


          batch_bar.update()
      batch_bar.close()

      train_loss = float(total_loss / len(train_loader))
      train_acc = 100 * num_correct / (len(train_loader) * run_config['batch_size'])
      lr = float(optimizer.param_groups[0]['lr'])

      print("Epoch {}/{}: Train Acc {:.04f}%, Train Loss {:.04f}, Learning Rate {:.04f}".format(
          epoch + 1,
          run_config['epochs'],
          train_acc ,
          train_loss,
          lr
          )
      )

      # What to log

      metrics = {
          "train_loss":train_loss,
          "train_acc": train_acc,
          'lr': lr
      }

      # Log to run
      wandb.log(metrics)

      # Updating the model version

      if train_acc > best_acc:
        best_acc = train_acc

        # Saving the model and optimizer states

        torch.save({
              'model_state_dict': model.state_dict(),
              'optimizer_state_dict': optimizer.state_dict()
              }, "Model.pth")

        # ALTERNATIVE 1: Saving Files as Artifacts
        # Creating Artifact
        model_artifact = wandb.Artifact(run_config['model'], type='model')

        # Adding model file to Artifact
        model_artifact.add_file("Model.pth")

        # Saving Artifact to WandB
        run.log_artifact(model_artifact)

        # ALTERNATIVE 2: Saving Files as Files
        wandb.save("Model.pth")

  if finish:
    wandb.finish()

## Simple Usage

You can run the training function and log the performance metrics of your choice into the WandB GUI. This simple method will allow you to monitor trends in a specefic run configuration as well as comparing different runs

In [11]:
run_config = {
    'model': '1-2dcnn',
    'optimizer':'sgd',
    'lr': 2e-3,
    'batch_size':64,
    'epochs': 5
}

train_loader, test_loader = build_data(run_config['batch_size'], data_train, data_test)

optimizer = get_optim(run_config['optimizer'], run_config['lr'], model)

criterion = nn.CrossEntropyLoss()

scaler = torch.cuda.amp.GradScaler()

/tmp/ipykernel_1182182/2711878366.py:15: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [12]:
run = wandb.init(
    #entity="wandb-starter",
    project="wandb-quickstart",
    #job_type="model-training",
    name=run_config['model'],
    config=run_config
    )

In [13]:
train(model)

Train:   0%|          | 0/782 [00:00<?, ?it/s]/tmp/ipykernel_1182182/2212453447.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_1182182/2212453447.py:24: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:822.)
  total_loss += float(loss)


Epoch 1/5: Train Acc 25.0460%, Train Loss 2.1039, Learning Rate 0.0020


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


Epoch 2/5: Train Acc 32.0692%, Train Loss 1.9492, Learning Rate 0.0020


Epoch 3/5: Train Acc 34.3290%, Train Loss 1.8866, Learning Rate 0.0020


Epoch 4/5: Train Acc 35.8476%, Train Loss 1.8470, Learning Rate 0.0020


Epoch 5/5: Train Acc 36.9765%, Train Loss 1.8153, Learning Rate 0.0020


lr,▁▁▁▁▁
train_acc,▁▅▆▇█
train_loss,█▄▃▂▁
lr,0.002
train_acc,36.9765
train_loss,1.81535


## Resume a previous run

In [14]:
RESUME_LOGGING = True ### Change to true to test the code for resuming an existing run

In [16]:
# Close a run left open by an earlier execution of this notebook.
if wandb.run is not None:
    wandb.finish()

project_name = "wandb-quickstart"
run_id = "4jtvb0pu"  # Replace with the ID of the run to resume
run_config_value = run_config if "run_config" in globals() else None
run_name = (run_config.get("model", "wandb-experiment")
            if isinstance(run_config_value, dict) else "wandb-experiment")

if RESUME_LOGGING:
    try:
        # Resume when possible, or create this run if it does not exist yet.
        run = wandb.init(
            id=run_id,
            resume="allow",
            project=project_name,
        )
    except Exception as exc:
        # A run ID cannot be shared by two active processes.
        error_text = str(exc).lower()
        resume_failed = (
            "run id" in error_text
            and ("in use" in error_text or "not been initialized" in error_text)
        )
        if not resume_failed:
            raise
        print(f"{run_id} is already in use; starting a new W&B run.")
        run = wandb.init(
            project=project_name,
            name=f"{run_name}-new",
            config=run_config_value,
            resume="never",
        )
else:
    run = wandb.init(
        project=project_name,
        name=run_name,
        config=run_config_value,
        resume="never",
    )

print(f"W&B run: {run.id}")
print(run.dir)


4jtvb0pu is already in use; starting a new W&B run.


W&B run: q9x4gsu0
/home/admin/CMU-11-785-s26/S26/documents/recitation_0/0.17/wandb/run-20260806_072438-q9x4gsu0/files


In [ ]:
### Test code to try appending metrics to previously logged metrics in the run
### Uncomment to try out

# test_new_metrics = {
#       "train_loss":1.5,
#       "train_acc": 40,
#       'lr': 0.001
#   }

# wandb.log(test_new_metrics)

## HyperParameter Sweeps


[Sweeps](https://docs.wandb.ai/guides/sweeps) are a way of automating hyperparameter tuning in Deep Learning Models. You set up the values that you want your sweep to try and then check the affect of changing each parameter on each value on the model.

In [17]:
# Initialize the sweep and set the method (grid, random or bayes"ian")

sweep_config = {
    'method': 'random'
    }

In [18]:
# What is the objective of the sweep (minimize loss, maximize accuracy)

metric = {
    'name':'loss',
    'goal':'minimize'
}
sweep_config['metric'] = metric

In [19]:
# Hyperparameters to work with

parameters_dict = {
    'optimizer':{
        'values': ['sgd', 'adam']
    },
    'learning_rate':{
        'distribution':'uniform',
        'min':2e-4,
        'max':1e-1
    },
    'batch_size': {
        'distribution': 'q_log_uniform_values',
        'q':4,
        'min': 16,
        'max': 128
    },
    'epochs':{
        'value': 5
    }
}
sweep_config['parameters'] = parameters_dict

In [20]:
# Initalizing the sweep

sweep_id = wandb.sweep(sweep_config, project="CIFAR-Sweep2")

Create sweep with ID: 4bglqye4
Sweep URL: https://wandb.ai/2792779840-zhejiang-university/CIFAR-Sweep2/sweeps/4bglqye4


In [21]:
def train_sweep(config = None):
    with wandb.init(config=config) as run:
        run.name=f"Jeel_{wandb.config.learning_rate}_{wandb.config.batch_size}_{wandb.config.optimizer}"
        config = wandb.config

        train_loader, test_loader = build_data(config.batch_size, data_train, data_test)

        model = Network().to(device)

        optimizer = get_optim(config.optimizer, config.learning_rate, model)

        criterion = nn.CrossEntropyLoss()

        scaler = torch.cuda.amp.GradScaler()

        for epoch in range(config.epochs):

            model, loss = train_epoch(model, train_loader, optimizer, criterion, scaler)

            wandb.log({'loss': loss})

In [22]:
# Running the sweep

wandb.agent(sweep_id, train_sweep, count=2)

wandb: 
wandb: Find logs at: wandb/run-20260806_072342-4jtvb0pu/logs


wandb: Agent Starting Run: 9edeucly with config:
wandb: 	batch_size: 28
wandb: 	epochs: 5
wandb: 	learning_rate: 0.05341533035784415
wandb: 	optimizer: adam
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


wandb: 
wandb: 🚀 View run 1-2dcnn-new at: https://wandb.ai/2792779840-zhejiang-university/wandb-quickstart/runs/q9x4gsu0
wandb: Find logs at: wandb/run-20260806_072438-q9x4gsu0/logs


/tmp/ipykernel_1182182/3451915675.py:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_1182182/53746039.py:11: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


loss,█▃▃▂▁
loss,1.60177


wandb: Agent Starting Run: lvim2ke1 with config:
wandb: 	batch_size: 24
wandb: 	epochs: 5
wandb: 	learning_rate: 0.06151560667194889
wandb: 	optimizer: sgd
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


/tmp/ipykernel_1182182/3451915675.py:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_1182182/53746039.py:11: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


loss,█▄▃▂▁
loss,1.21809


## Artifact and Model Versioning

Artifacts are a method of managing versions for data and models. You can use the artifacts to keep and compare versions of your model while training making it easier to share data and models between team members.

In [23]:
run_config = {
    'model': '1-2dcnn',
    'optimizer':'adam',
    'lr': 5e-3,
    'batch_size':20,
    'epochs': 5
}

train_loader, test_loader = build_data(run_config['batch_size'], data_train, data_test)
optimizer = get_optim(run_config['optimizer'], run_config['lr'], model)
criterion = nn.CrossEntropyLoss()
scaler = torch.cuda.amp.GradScaler()

/tmp/ipykernel_1182182/3016575026.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [24]:
run = wandb.init(
    project="wandb-quickstart",
    job_type="model-training",
    name=run_config['model'],
    config=run_config
    )

In [25]:
train(model,finish= False) #run should not finish for using artifact

Train:   0%|          | 0/2500 [00:00<?, ?it/s]/tmp/ipykernel_1182182/2212453447.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 1/5: Train Acc 44.6840%, Train Loss 1.5479, Learning Rate 0.0050


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


Epoch 2/5: Train Acc 53.1780%, Train Loss 1.3333, Learning Rate 0.0050


Epoch 3/5: Train Acc 57.7340%, Train Loss 1.2207, Learning Rate 0.0050


Epoch 4/5: Train Acc 59.3360%, Train Loss 1.1732, Learning Rate 0.0050


Epoch 5/5: Train Acc 60.3660%, Train Loss 1.1449, Learning Rate 0.0050


In [27]:
## Retreiving the model

# Getting the latest version of the artifact
artifact = run.use_artifact('{}:latest'.format(run_config['model']))
# Downloading the artifact
artifact_dir = artifact.download()
# Loading the model. The training cell stores the artifact as Model.pth.
model_path = os.path.join(artifact_dir, 'Model.pth')
if not os.path.isfile(model_path):
    raise FileNotFoundError(f'Expected artifact file not found: {model_path}')
model_dict = torch.load(model_path, map_location='cpu')



# Loading weights
model.load_state_dict(model_dict['model_state_dict'])
# Loading optimizer state
optimizer.load_state_dict(model_dict['optimizer_state_dict'])

wandb:   1 of 1 files downloaded.  


In [28]:
# Finishing runs
wandb.finish()

lr,▁▁▁▁▁
train_acc,▁▅▇██
train_loss,█▄▂▁▁
lr,0.005
train_acc,60.366
train_loss,1.14495
